#Organoid Analysis

Data Exploration

In [3]:
#Importing Required Libaries And Setting Up Visualization Styles
import numpy as np,pandas as pd
import matplotlib.pyplot as plt,seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"]=(12,6)
plt.rcParams["font.size"]=10

In [ ]:
#Loading The Datasets
def load(dir="data"):
  path=Path(dir)
  xlfile=list(path.glob("*.xlsx"))+list(path.glob("*.xls"))
  all_data={}
  for i in xlfile:
    fname=i.stem  #Get Filename Without Extension
    print(f"Loading: {i.name}\n")
    try:
      xl_f=pd.ExcelFile(i)
      sheet=xl_f.sheet_names
      print(f"Found ->{len(sheet)} Sheet(s): {sheet}")
      all_data[fname]={}
      for j in sheet:
        df=pd.read_excel(i,sheet_name=j)
        all_data[fname][j]=df
        print(f"Sheet:'{j}':{df.shape[0]} Rows \u00d7 {df.shape[1]} Columns")
    except Exception as e:
      print(f"Error Loading {i.name}:{str(e)}")
  return all_data

dataset=load("/content/drive/MyDrive/Organoid Analysis")
print(f"\n Loaded {len(dataset)} Excel Files Successfully")

In [ ]:
#Cleaning Up The Files
def clean_filter(all_data):
    clean={}
    for i,j in all_data.items():
        clean[i]={}
        for k,df in j.items():
            if df.shape[0]==0:
                print(f"Skipping empty sheet: {i}::{k}")
                continue
            df=df.dropna(axis=1,how="all")
            clean[i][k]=df
    return clean

dataset=clean_filter(dataset)
print(f"\n Cleaning Complete")

In [ ]:
#Exploring The Datasets
def explore(df,dataset_name):
    print(f"Exploring Dataset: {dataset_name}\n")
    print(f"\nDataset Dimensions: {df.shape[0]} Rows \u00d7 {df.shape[1]} Columns")

    #Identify potential hierarchy columns
    hier_keywords=["batch","sample","well","field","condition","treatment"]
    hier_cols=[
        i for i in df.columns if any(j in i.lower() for j in hier_keywords)
        ]
    if hier_cols:
        print(f"\n Hierarchy Column Detected:")
        for i in hier_cols:
            unique_vals=df[i].nunique()
            print(f"{i}:{unique_vals} Unique Value(s)")
            if unique_vals<=10:
                unique_values=df[i].dropna().unique()
                try:
                    if len(unique_values)>0:
                        #Converting To String For Sorting
                        sorted_vals=sorted([str(val) for val in unique_values])
                except Exception as e:
                    print(f"Values: {[str(val) for val in unique_values]}")
    else:
        print("\n No Obvious Hierarchy Columns Detected")
    #Printing Column Data Types
    print(f"\n Column Data Types:")
    print(df.dtypes.value_counts())

    return hier_cols

if not dataset: # Added this check
    print("The 'dataset' variable is empty. Please check the data loading step (Cell OJDKhAiNtd7l) to ensure data is loaded correctly.")
else:
    for i,j in dataset.items():
        for k,df in j.items():
            dataset_id=f"{i}::{k}"
            explore(df,dataset_id)

In [ ]:
#Creating An Inventory For All Our Datasets
def create_inventory(all_data):
    inventory=[]
    for filename,sheets in all_data.items():
        for sheet,df in sheets.items():
            #Counting Numeric Vs Non-numeric Columns
            numeric_cols=df.select_dtypes(include=[np.number]).columns
            non_numeric_cols=df.select_dtypes(exclude=[np.number]).columns

            #Identifying Unique wells/samples If Present
            well_col=[i for i in df.columns if "well" in i.lower()]
            n_wells=df[well_col[0]].nunique() if well_col else "N/A"

            inventory.append(
                {
                    "File":filename,
                    "Sheet":sheet,
                    "Rows (Cells)":df.shape[0],
                    "Total Columns":df.shape[1],
                    "Numeric Features":len(numeric_cols),
                    "Non-Numeric Columns":len(non_numeric_cols),
                    "Unique Wells":n_wells,
                }
            )
    inventory_df=pd.DataFrame(inventory)
    return inventory_df

inventory=create_inventory(dataset)
print("Dataset Inventory Summary")
print(inventory.to_string(index=False))

In [ ]:
#Accessing The Data Quality

def access_qlty(df,dataset_name):
    print(f"Data Quality Assessment Of:{dataset_name}")
    #For Missing values
    miss=df.isnull().sum()
    miss_pct=(miss/len(df))*100
    miss_df=pd.DataFrame(
        {
            "Column": miss.index,
            "Missing_Count": miss.values,
            "Missing_Percent": miss_pct.values,
        }
    )
    miss_df=miss_df[miss_df["Missing_Count"]>0].sort_values(
        "Missing_Percent",ascending=False
    )
    if len(miss_df)>0:
        print(f"\nMissing Values Detected:")
        print(miss_df.to_string(index=False))
    else:
        print(f"\nNo Missing Values Detected")
    #Checking Numeric Columns For Potential Issues
    numeric_cols=df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols)>0:
        print(f"\nNumeric Features Summary:")
        print(f"Total Numeric Columns: {len(numeric_cols)}")
        #Checking For Columns With Zero Variance (All Same Value)
        var0_cols=[i for i in numeric_cols if df[i].std()==0]
        if var0_cols:
            print(f"\nColumns With Zero Variance (Constant Values):")
            for i in var0_cols:
                print(f"{i}: Value={df[i].iloc[0]}")
        else:
            print("All Numeric Columns Have Variance")
    #Sample Sizes Per Condition (If Applicable)
    condn_cols=[
        i for i in df.columns
        if any(j in i.lower() for j in ["condition","treatment","well"])
    ]
    if condn_cols:
        print(f"\nSample Sizes Per Condition:")
        for i in condn_cols[:2]:
            print(f"\n{i}:")
            print(df[i].value_counts().to_string())

    return miss_df

for filename,sheets in dataset.items():
    for sheet,df in sheets.items():
        dataset_id=f"{filename} - {sheet}"
        access_qlty(df,dataset_id)

In [ ]:
#Checking For Outliers
def detect_outliers(df,dataset_name,shown_feat=5):
    print(f"Outlier Detection For: {dataset_name}")
    numeric_cols=df.select_dtypes(include=[np.number]).columns
    outlier_summary=[]
    for i in numeric_cols:
        Q1=df[i].quantile(0.25)
        Q3=df[i].quantile(0.75)
        IQR=Q3-Q1
        low=Q1-1.5*IQR
        high=Q3+1.5*IQR
        outliers=df[(df[i]<low) | (df[i]>high)]
        n_outliers=len(outliers)
        pct_outliers=(n_outliers/len(df))*100

        outlier_summary.append(
            {
                "Feature":i,
                "N_Outliers":n_outliers,
                "Percent_Outliers":pct_outliers,
                "Lower_Bound":low,
                "Upper_Bound":high,
            }
        )
    outlier_df=pd.DataFrame(outlier_summary)
    outlier_df=outlier_df[outlier_df["N_Outliers"]>0].sort_values(
        "Percent_Outliers",ascending=False
    )
    if len(outlier_df)>0:
        print(f"\nOutlier Detected (IQR Method):")
        print(f"\nTop {shown_feat} Features With Most Outliers:")
        print(outlier_df.head(shown_feat).to_string(index=False))
    else:
        print("\nNo Outliers Detected Using IQR Method")
    return outlier_df

for filename,sheets in dataset.items():
    for sheet,df in sheets.items():
        if df.shape[0]>0:  #Only If Dataset Isn't Empty
            dataset_id=f"{filename} -> {sheet}"
            detect_outliers(df,dataset_id)

Visualizing Feature Distributions

In [ ]:
def plot_distr(df,dataset_name,max_feat=6):
    numeric_cols=df.select_dtypes(include=[np.number]).columns
    #Selecting Features To Plot
    pltfeat=numeric_cols[:max_feat]
    n_features=len(pltfeat)
    n_rows=(n_features+2)//3
    n_cols=min(3,n_features)
    fig,axes=plt.subplots(n_rows,n_cols,figsize=(15,5*n_rows))
    fig.suptitle(f"Feature Distributions: {dataset_name}",fontsize=16,y=1.02)

    #Flattening Axes For Easier Iteration
    if n_features==1:
        axes=[axes]
    elif n_features>1:
        axes=axes.flatten() if n_rows>1 else [axes] if n_cols==1 else axes

    for i,j in enumerate(pltfeat):
        ax=axes[i] if n_features>1 else axes[0]

        #Removing Infinite Values And NaN For Plotting
        data=df[j].replace([np.inf,-np.inf],np.nan).dropna()
        if len(data)>0:
            ax.hist(data,bins=30,edgecolor="black",alpha=0.7)
            ax.set_xlabel(j,fontsize=10)
            ax.set_ylabel("Frequency",fontsize=10)
            ax.set_title(f"{j}\n(n={len(data)},\u03bc={data.mean():.2f})",fontsize=9)
            ax.grid(True,alpha=0.3)
    #Hiding Unused Subplots
    for i in range(n_features,len(axes)):
        axes[i].set_visible(False)
    plt.tight_layout()
    plt.show()

# Removed key_datasets filter to plot distributions for all loaded datasets
for filename,sheets in dataset.items():
    for sheet,df in sheets.items():
        if df.shape[0]>0:  # Only If Dataframe Has Data
            dataset_id=f"{filename} -> {sheet}"
            plot_distr(df,dataset_id)

Performing Statistical Evaluation

In [ ]:
def analyze_stats(df,dataset_name):
    from scipy import stats
    numeric_cols=df.select_dtypes(include=[np.number]).columns
    dist_stats=[]
    for i in numeric_cols:
        data=df[i].replace([np.inf,-np.inf],np.nan).dropna()
        if len(data)>0:
            dist_stats.append(
                {
                    "Feature":i,
                    "Mean":data.mean(),
                    "Median":data.median(),
                    "Std":data.std(),
                    "Skewness":stats.skew(data),
                    "Kurtosis":stats.kurtosis(data),
                }
            )
    dist_df=pd.DataFrame(dist_stats)

    #Identifying Highly Skewed Features (|Skewness|>1)
    highly_skewed=dist_df[abs(dist_df["Skewness"])>1].sort_values(
        "Skewness",key=abs,ascending=False
    )
    if len(highly_skewed)>0:
        print(f"\nHighly Skewed Features:")
        print(highly_skewed[["Feature","Skewness"]].head(10).to_string(index=False))
    else:
        print("\nNo Highly Skewed Features Detected")
    return dist_df

# Removed key_datasets filter to analyze stats for all loaded datasets
for filename,sheets in dataset.items():
    for sheet,df in sheets.items():
        if df.shape[0]>0:
            dataset_id=f"{filename} -> {sheet}"
            analyze_stats(df,dataset_id)

Checking For Relationships/Correlation Between Features

In [ ]:
def calc_corr_mat(df,dataset_name,threshold=0.7):
    print(f"Feature Correlation For: {dataset_name}")
    numeric_cols=df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols)<2:
        print("\nNot Enough Numeric Features For Correlation Analysis")
        return None
    #Calculating Correlation Matrix
    corr_mat=df[numeric_cols].corr()
    #Finding Highly Correlated Pairs
    high_corr_pairs=[]
    for i in range(len(corr_mat.columns)):
        for j in range(i+1,len(corr_mat.columns)):
            corr_val=corr_mat.iloc[i, j]
            if abs(corr_val)>=threshold:
                high_corr_pairs.append(
                    {
                        "Feature_1":corr_mat.columns[i],
                        "Feature_2":corr_mat.columns[j], # Corrected typo from corr_matr
                        "Correlation":corr_val,
                    }
                )
    if high_corr_pairs:
        print(f"\nHighly Correlated Feature Pairs (|r|\u2265{threshold}):")
        corr_df=pd.DataFrame(high_corr_pairs).sort_values(
            "Correlation",key=abs,ascending=False
        )
        print(corr_df.to_string(index=False))
    else:
        print(f"\nNo Feature Pairs With |Correlation|\u2265{threshold}")
    return corr_mat

corr_mats={}
# Removed key_datasets filter to calculate correlation for all loaded datasets
for filename,sheets in dataset.items():
    for sheet,df in sheets.items():
        if df.shape[0]>0:
            dataset_id=f"{filename} -> {sheet}"
            corr_mat=calc_corr_mat(df,dataset_id)
            if corr_mat is not None:
                corr_mats[dataset_id]=corr_mat

Visualizing The Correlations

In [ ]:

def plot_corr_heatmap(corr_mat,dataset_name,figsize=(12,10)):
    if corr_mat is None or len(corr_mat)<2:
        print(f"Skipping Heatmap For {dataset_name} -> Insufficient Data")
        return
    #Limiting To Reasonable Number Of Features For Visualization
    if len(corr_mat)>20:
        print(f"Too Many Features ({len(corr_mat)}) -> Showing Top 20 By Variance")
        #Selecting Features With Highest Variance
        vars=corr_mat.var().sort_values(ascending=False)
        top_feat=vars.head(20).index
        corr_mat=corr_mat.loc[top_feat,top_feat]

    plt.figure(figsize=figsize)
    sns.heatmap(
        corr_mat,
        annot=False,
        cmap="coolwarm",
        center=0,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink":0.8},
        vmin=-1,
        vmax=1,
    )
    plt.title(f"Correlation Matrix: {dataset_name}",fontsize=14,pad=20)
    plt.xticks(rotation=45,ha="right",fontsize=8)
    plt.yticks(rotation=0,fontsize=8)
    plt.tight_layout()
    fname_safe=dataset_name.replace(" - ","_").replace(" ","_")
    plt.show()

for i,corr_mat in corr_mats.items():
    plot_corr_heatmap(corr_mat,i)

Comparing The Datasets

In [ ]:
def compare_feats(all_data):
    #Extracting All Feature Sets
    dataset_feats={}
    for filename,sheets in all_data.items():
        for sheet,df in sheets.items():
            dataset_id=f"{filename} -> {sheet}"
            #Getting Numeric Feature Names
            feats=set(df.select_dtypes(include=[np.number]).columns)
            dataset_feats[dataset_id]=feats

    #Finding Common Features Across All Datasets
    all_feat_sets=list(dataset_feats.values())
    com_feats=set.intersection(*all_feat_sets) if all_feat_sets else set()
    print(f"\nFeatures Common To All Dataset ({len(com_feats)}):")
    if com_feats:
        for i in sorted(com_feats):
            print(f"{i}")
    else:
        print("(No Features Common To All Datasets)")

    #Finding Unique Features In Each Dataset
    print(f"\nUnique Features Per Dataset:")
    for dataset_id,feats in dataset_feats.items():
        other_feats=set()
        for other_id,other_feats in dataset_feats.items():
            if other_id!=dataset_id:
                other_feats.update(other_feats)
        unique=feats-other_feats
        if unique:
            print(f"\n{dataset_id}:")
            print(f"Unique Features ({len(unique)}): {sorted(unique)}")

    #Checking For Potential Duplicates (Same Features And Same Row Count)
    print(f"\nChecking For Duplicate Datasets:")
    chkd_pairs=set()
    dups_found=False
    for id1,f1 in dataset_feats.items():
        for id2,f2 in dataset_feats.items():
            if id1<id2 and (id1,id2) not in chkd_pairs:
                chkd_pairs.add((id1,id2))
                #Checking If Features Match
                if f1==f2:
                    #Getting Dataframes And Checking Row Counts
                    df1=all_data[id1.split(" - ")[0]][id1.split(" - ")[1]]
                    df2=all_data[id2.split(" - ")[0]][id2.split(" - ")[1]]
                    if df1.shape[0]==df2.shape[0]:
                        print(f"\nPotential Duplicate Detected:")
                        print(f"{id1}")
                        print(f"{id2}")
                        print(
                            f"Both Have {df1.shape[0]} Rows And {len(f1)} Features"
                        )
                        dups_found=True
    if not dups_found:
        print("No Duplicate Datasets Detected")
    return dataset_feats

dataset_feats=compare_feats(dataset)

Creating A Summary Table For Showing Key Statistics For Each Dataset Side By Side

In [ ]:
def create_summary(all_data):
    summary_data=[]
    for filename,sheets in all_data.items():
        for sheet_name,df in sheets.items():
            numeric_cols = df.select_dtypes(include=[np.number]).columns

            #Identifying Hierarchy Columns
            well_col=[i for i in df.columns if "well" in i.lower()]
            field_col=[i for i in df.columns if "field" in i.lower()]

            n_wells=df[well_col[0]].nunique() if well_col else "N/A"
            n_fields=df[field_col[0]].nunique() if field_col else "N/A"

            #Calculating Missing Data Percentage
            miss_pct=(df.isnull().sum().sum()/(df.shape[0]*df.shape[1]))*100

            summary_data.append(
                {
                    "Dataset":f"{filename} -> {sheet_name}",
                    "Total_Cells":df.shape[0],
                    "Total_Features":df.shape[1],
                    "Numeric_Features":len(numeric_cols),
                    "N_Wells":n_wells,
                    "N_Fields":n_fields,
                    "Missing_Data_%":f"{miss_pct:.2f}",
                    "Suggested_Use":"", #Manual Entry Based On Knowledge
                }
            )

    summary_df=pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))

    #Saving As CSV For Reference
    summary_df.to_csv("dataset_summary.csv", index=False)
    print(f"\nSummary Saved To 'dataset_summary.csv'")
    return summary_df

summary_df=create_summary(dataset)